# R10B — Paper Risk / Halt Filter Reconstruction

Paper risk-management ablation only: the underlying NN v1/NN v2 BCE model, deterministic TOP2/TOP3 ranking, and R10A execution/accounting/allocation formulas (equal-weight and paper wins/buys/streak weight, both carried forward per `CARRY_FORWARD_BOTH_FOR_R10B`) are all reused verbatim -- this notebook tests only whether 5 source-derived loss-mitigation rules (Table 2, Bock & Maewal SSRN 4280640), applied as an eligibility filter ahead of Top-K selection with backfill, improve the R10A baseline. Verified before anything else: with all 5 filters disabled, R10B reproduces R10A's ending equity and trade counts exactly. See `docs/weekly_sector_rotation_r10b_risk_filters.md` for the full write-up and `r10b_risk_filters.py` for the shared, unit-tested filter-state-machine implementation this notebook imports. Does not implement the FINAL_INTEGRATED_STRATEGY_PERFORMANCE_EVALUATION.

In [1]:
"""
R10B full pipeline (scratchpad validation run). Once verified, transcribed
into r10b_risk_filter_ablation.ipynb and executed there for the committed
artifacts. Paper risk/halt-filter ablation only: the underlying NN v1/v2
BCE model, TOP2/TOP3 ranking, and R10A execution/accounting/allocation
formulas are all reused verbatim -- R10B only tests whether 5 source-derived
loss-mitigation rules, applied as an eligibility filter ahead of Top-K
selection (with backfill), improve the R10A baseline.
"""
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()
REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

sys.path.insert(0, str(REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation"))
import r7_financial_losses as r7  # noqa: E402
import r10a_portfolio as r10a  # noqa: E402
import r10b_risk_filters as r10b  # noqa: E402

SECTOR_ETFS = r7.SECTOR_ETFS
BENCHMARK_ETF = r7.BENCHMARK_ETF
FULL_FEATURE_COLS = r7.FULL_FEATURE_COLS

# ---------------------------------------------------------------------------

## 1. Load panel + R1 raw execution prices, authoritative feature set, and reproduce/verify R1's exact fold schedule.

In [2]:
# 1. Load panel + R1 raw execution prices, authoritative feature set, fold
#    schedule -- verbatim R1/R7/R8/R9/R10A.
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_first_open",
                "next_week_final_close", "next_week_open_to_close_return", "next_week_vti_return"]],
    on=["date", "symbol"], how="left",
)
assert set(FULL_FEATURE_COLS).issubset(set(FEATURE_COLUMNS))

md = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md["label_binary"] = md["target_abs_1pct_next_week"].astype(int)
md["year"] = md["date"].dt.year
years_all = sorted(md["year"].unique())
assert years_all == list(range(2005, 2027))
test_years = r7.derive_r1_fold_schedule(years_all)
assert test_years == list(range(2008, 2027)), f"fold schedule diverges from validated R1 schedule: {test_years}"
print(f"[{time.time()-t0:.0f}s] data loaded, {len(md)} rows, test_years={test_years[0]}-{test_years[-1]}")

# ---------------------------------------------------------------------------

[0s] data loaded, 12253 rows, test_years=2008-2026


## 2. VTI execution-aligned benchmark -- verbatim R10A.

In [3]:
# 2. VTI execution-aligned benchmark -- verbatim R10A.
# ---------------------------------------------------------------------------
week_bounds = md[["date", "next_week_first_trading_date", "next_week_last_trading_date"]].drop_duplicates(subset=["date"])
vti_daily = pd.read_csv(PRICES_ROOT / f"{BENCHMARK_ETF}.csv", parse_dates=["date"])[["date", "open", "close"]]
vti_open_lookup = vti_daily.set_index("date")["open"]
vti_close_lookup = vti_daily.set_index("date")["close"]
vti_bench_rows = []
for _, row in week_bounds.iterrows():
    entry_date, exit_date = row["next_week_first_trading_date"], row["next_week_last_trading_date"]
    if pd.isna(entry_date) or pd.isna(exit_date):
        continue
    if entry_date not in vti_open_lookup.index or exit_date not in vti_close_lookup.index:
        continue
    entry_open = vti_open_lookup.loc[entry_date]; exit_close = vti_close_lookup.loc[exit_date]
    vti_bench_rows.append({"date": row["date"], "vti_trade_return": r10a.trade_return(entry_open, exit_close)})
vti_benchmark = pd.DataFrame(vti_bench_rows).set_index("date")
vti_weekly_for_metrics = vti_benchmark.reset_index().rename(columns={"vti_trade_return": "portfolio_return"})
vti_weekly_for_metrics["equity_after"] = r10a.equity_curve_from_returns(
    vti_weekly_for_metrics["portfolio_return"].to_numpy())[1:]
print(f"[{time.time()-t0:.0f}s] Step 2 (VTI benchmark) done, {len(vti_benchmark)} weeks.")

# ---------------------------------------------------------------------------

[0s] Step 2 (VTI benchmark) done, 1114 weeks.


## 3. Model definitions + walk-forward training -- verbatim R1/R7/R8/R9/R10A cold-restart cadence -- then verify deterministic pooled AUC reproduces the accepted values exactly.

In [4]:
# 3. Model definitions + walk-forward training -- verbatim R1/R7/R8/R9/R10A.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


BCE_LOSS = nn.BCEWithLogitsLoss()


def train_one_model(X_fit, y_fit, X_val, y_val, n_features, seed, v2=False):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features) if v2 else SmallMLP(n_features)
    lr = V2_LEARNING_RATE if v2 else LEARNING_RATE
    wd = V2_WEIGHT_DECAY if v2 else WEIGHT_DECAY
    max_epochs = V2_MAX_EPOCHS if v2 else MAX_EPOCHS
    patience = V2_PATIENCE if v2 else PATIENCE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(max_epochs):
        model.train(); optimizer.zero_grad()
        loss = BCE_LOSS(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = BCE_LOSS(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def walk_forward_bce(md, years_, test_years_, feature_cols, v2=False):
    oof_frames = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()

        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_one_model(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset, v2=True)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_one_model(X_fit, y_fit, X_val, y_val, len(feature_cols), NN_SEED, v2=False)
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = md.loc[test_mask, ["date", "symbol", "next_week_first_open", "next_week_final_close",
                                   "next_week_open_to_close_return", "next_week_vti_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
    return pd.concat(oof_frames, ignore_index=True)


oof_v1 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=False)
auc_v1 = roc_auc_score(oof_v1.label_binary, oof_v1.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v1 deterministic pooled_auc={auc_v1:.6f} (expected 0.569368)")
assert abs(auc_v1 - 0.5693678469913763) < 1e-4, "NN v1 AUC does not reproduce accepted value"

oof_v2 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=True)
auc_v2 = roc_auc_score(oof_v2.label_binary, oof_v2.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v2 deterministic pooled_auc={auc_v2:.6f} (expected 0.570832)")
assert abs(auc_v2 - 0.5708322204557499) < 1e-4, "NN v2 AUC does not reproduce accepted value"
print(f"[{time.time()-t0:.0f}s] Step 3 (walk-forward training, both models) done.")

# ---------------------------------------------------------------------------

[4s] NN_v1 deterministic pooled_auc=0.569368 (expected 0.569368)


[19s] NN_v2 deterministic pooled_auc=0.570832 (expected 0.570832)
[19s] Step 3 (walk-forward training, both models) done.


## 4. Build the FULL 11-sector weekly ranking table per model (all 11 ranks, not just Top-K) -- required for eligibility-filter backfill (a blocked rank-2 IS backfilled by rank-3, per the paper's own algorithm ordering).

In [5]:
# 4. Build FULL 11-sector weekly ranking table per model (all ranks, needed
#    for backfill -- unlike R8/R9/R10A which only kept Top-K).
# ---------------------------------------------------------------------------
OOF_BY_MODEL = {"NN_v1": oof_v1, "NN_v2": oof_v2}
K_VALUES = [2, 3]


def build_full_ranking(oof: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for date, g in oof.groupby("date"):
        if len(g) < 11:
            continue
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        gg = g.copy(); gg["rank"] = ranked.astype(int).to_numpy()
        rows.append(gg)
    return pd.concat(rows, ignore_index=True).sort_values(["date", "rank"]).reset_index(drop=True)


full_ranking = {model_name: build_full_ranking(oof) for model_name, oof in OOF_BY_MODEL.items()}
print(f"[{time.time()-t0:.0f}s] Step 4 (full 11-sector weekly ranking, both models) done.")

# ---------------------------------------------------------------------------

[19s] Step 4 (full 11-sector weekly ranking, both models) done.


## 5. Simulation engine: filter-triggering state (Rules A-E) is always driven by GROSS returns/equity (matching R10A's own precedent for allocation state), so gross and cost-adjusted settlement are two views of the identical trade/halt sequence.

In [6]:
# 5. Simulation engine. Filter-triggering state (A/B/C/D/E) is ALWAYS driven
#    by GROSS trade returns/equity, exactly matching R10A's own precedent
#    that allocation state (wins/buys/streak) is gross-based regardless of
#    the cost/no-cost settlement being reported -- so a single simulation
#    produces the identical sequence of trades/halts for both the gross and
#    cost-adjusted variants; only the SETTLEMENT differs (reusing R10A's
#    own `settle_fractional` on two different return columns).
# ---------------------------------------------------------------------------
FILTER_CONFIGS = {
    "BASE": frozenset(), "A": frozenset("A"), "B": frozenset("B"), "C": frozenset("C"),
    "D": frozenset("D"), "E": frozenset("E"),
    "AB": frozenset("AB"), "ABC": frozenset("ABC"), "ABCD": frozenset("ABCD"), "ABCDE": frozenset("ABCDE"),
}
INDIVIDUAL_CONFIG_NAMES = ["BASE", "A", "B", "C", "D", "E"]
CUMULATIVE_CONFIG_NAMES = ["BASE", "A", "AB", "ABC", "ABCD", "ABCDE"]


def simulate(full_rank: pd.DataFrame, k: int, allocation_method: str, active_filters: frozenset,
             build_trigger_log: bool = False):
    engine = r10b.RiskFilterEngine(symbols=SECTOR_ETFS, active_filters=active_filters)
    alloc_tracker = r10a.AllocationStateTracker(SECTOR_ETFS)
    dates_sorted = sorted(full_rank["date"].unique())

    ledger_rows = []
    trigger_rows = []
    equity_running = r10a.STARTING_CAPITAL  # only used to seed Rule C's equity_before/after tracking

    for week_idx, date in enumerate(dates_sorted):
        week = full_rank[full_rank["date"] == date].sort_values("rank")
        month = date.month
        year = date.year
        ranked_symbols = week["symbol"].tolist()

        halted = engine.is_halted(week_idx)
        eligible = set() if halted else engine.eligible_symbols(week_idx, month)
        selected = [] if halted else engine.select_top_k_from_eligible(ranked_symbols, eligible, k)

        if build_trigger_log:
            eligible_before_any = set(ranked_symbols)  # informational: full candidate set this week
            for _, r in week.iterrows():
                s = r["symbol"]
                trigger_rows.append({
                    "date": date, "symbol": s, "original_rank": int(r["rank"]),
                    "eligible_before_filters": s in eligible_before_any,
                    "filter_A_block": ("A" in active_filters) and (engine.recent_loss_block_at.get(s) == week_idx),
                    "filter_B_portfolio_halt": ("B" in active_filters) and (week_idx in engine.halted_week_idx),
                    "filter_C_portfolio_halt": ("C" in active_filters) and (week_idx in engine.halted_week_idx),
                    "filter_D_block": ("D" in active_filters) and engine.symbol_permanently_blocked.get(s, False),
                    "filter_E_block": ("E" in active_filters) and r10b.is_q4(month)
                                       and engine.symbol_buys.get(s, 0) > 0
                                       and (engine.symbol_wins.get(s, 0) / max(engine.symbol_buys.get(s, 1), 1)) < r10b.MIN_SYMBOL_WIN_RATE_THRESHOLD,
                    "final_selected": s in selected,
                })

        if selected:
            if allocation_method == "EQUAL_WEIGHT":
                weights = {s: r10a.equal_weight(len(selected)) for s in selected}
            else:
                raw = {s: alloc_tracker.raw_weight_for(s) for s in selected}
                weights = r10a.normalize_paper_weights(raw)
        else:
            weights = {}

        trades = {}
        gross_portfolio_return = 0.0
        for _, r in week[week["symbol"].isin(selected)].iterrows():
            s = r["symbol"]; w = weights[s]
            gross_ret = r["next_week_open_to_close_return"]
            trades[s] = gross_ret
            gross_portfolio_return += w * gross_ret
            ledger_rows.append({
                "date": date, "symbol": s, "rank": int(r["rank"]), "weight": w,
                "entry_open": r["next_week_first_open"], "exit_close": r["next_week_final_close"],
                "gross_trade_return": gross_ret, "cost_adjusted_trade_return": r10a.apply_round_trip_cost(gross_ret),
                "vti_trade_return": r["next_week_vti_return"], "n_selected_this_week": len(selected),
                "halted": halted,
            })

        equity_before = equity_running
        equity_after = equity_before * (1 + gross_portfolio_return)
        portfolio_dollar_pnl = equity_after - equity_before

        for s in selected:
            alloc_tracker.record_trade(s, trades[s])
        engine.record_week_outcome(week_idx, month, year, trades, portfolio_dollar_pnl, equity_before, equity_after)
        equity_running = equity_after

    ledger = pd.DataFrame(ledger_rows)
    trigger_log = pd.DataFrame(trigger_rows) if build_trigger_log else None
    return ledger, trigger_log, dates_sorted, engine


print(f"[{time.time()-t0:.0f}s] Step 5 (simulation engine) defined.")

# ---------------------------------------------------------------------------

[19s] Step 5 (simulation engine) defined.


## 6. CRITICAL CHECKPOINT (Step 3): verify R10B with all 5 filters disabled reproduces R10A exactly -- ending equity and total trade count, all 8 baseline variants.

In [7]:
# 6. Step 3 verification: R10B with all filters off MUST reproduce R10A
#    exactly (weekly positions, weekly returns, ending equity, CAGR, max
#    drawdown, total trades).
# ---------------------------------------------------------------------------
r10a_performance = pd.read_csv(OUTPUTS_ROOT / "r10a_performance_summary.csv")

for model_name in ["NN_v1", "NN_v2"]:
    for k in K_VALUES:
        for alloc in r10a.ALLOCATION_METHODS:
            ledger, _, dates_sorted, _ = simulate(full_ranking[model_name], k, alloc, FILTER_CONFIGS["BASE"])
            weekly = r10a.equity_curve_from_returns  # noqa
            settled = None
            weekly_ret = ledger.groupby("date").apply(lambda g: (g["weight"] * g["gross_trade_return"]).sum(),
                                                        include_groups=False).rename("portfolio_return").reset_index()
            weekly_ret = weekly_ret.sort_values("date").reset_index(drop=True)
            equity = r10a.equity_curve_from_returns(weekly_ret["portfolio_return"].to_numpy())
            ending_equity = equity[-1]
            r10a_row = r10a_performance[(r10a_performance.model == model_name) & (r10a_performance.K == k) &
                                         (r10a_performance.allocation_method == alloc)].iloc[0]
            diff = abs(ending_equity - r10a_row["ending_capital"])
            assert diff < 1.0, (f"R10B BASE does not reproduce R10A for {model_name} K={k} {alloc}: "
                                 f"R10B={ending_equity:.2f} R10A={r10a_row['ending_capital']:.2f} diff={diff:.4f}")
            assert len(ledger) == r10a_row["total_etf_trades"], \
                f"trade count mismatch {model_name} K={k} {alloc}: {len(ledger)} vs {r10a_row['total_etf_trades']}"
print(f"[{time.time()-t0:.0f}s] Step 6 (R10B BASE reproduces R10A exactly, all 8 variants) VERIFIED.")

# ---------------------------------------------------------------------------

[23s] Step 6 (R10B BASE reproduces R10A exactly, all 8 variants) VERIFIED.


## 7. Run all 10 unique filter configurations (6 individual: BASE,A,B,C,D,E + 6 cumulative: BASE,A,AB,ABC,ABCD,ABCDE) x 8 baseline variants = 80 simulations.

In [8]:
# 7. Run all 10 unique filter configs x 8 baseline variants. Build the
#    per-candidate trigger log only for cumulative configs (Step 16).
# ---------------------------------------------------------------------------
all_ledgers = {}
all_trigger_logs = {}
all_engines = {}
for model_name in ["NN_v1", "NN_v2"]:
    for k in K_VALUES:
        for alloc in r10a.ALLOCATION_METHODS:
            for config_name, filters in FILTER_CONFIGS.items():
                need_log = config_name in CUMULATIVE_CONFIG_NAMES
                ledger, trigger_log, _, engine = simulate(full_ranking[model_name], k, alloc, filters,
                                                           build_trigger_log=need_log)
                all_ledgers[(model_name, k, alloc, config_name)] = ledger
                all_engines[(model_name, k, alloc, config_name)] = engine
                if need_log:
                    all_trigger_logs[(model_name, k, alloc, config_name)] = trigger_log
    print(f"[{time.time()-t0:.0f}s] {model_name}: all filter configs simulated (both K, both allocations)")
print(f"[{time.time()-t0:.0f}s] Step 7 (all 8 variants x 10 filter configs) done, "
      f"{len(all_ledgers)} simulations total.")

# ---------------------------------------------------------------------------

[43s] NN_v1: all filter configs simulated (both K, both allocations)


[63s] NN_v2: all filter configs simulated (both K, both allocations)
[63s] Step 7 (all 8 variants x 10 filter configs) done, 80 simulations total.


## 8. Gross + cost-adjusted (10bps round-trip) performance metrics for every one of the 80 simulations.

In [9]:
# 8. Performance metrics (gross + cost-adjusted) for every simulation.
# ---------------------------------------------------------------------------
def compute_core_metrics(weekly: pd.DataFrame, starting_capital: float = r10a.STARTING_CAPITAL) -> dict:
    returns = weekly["portfolio_return"].to_numpy()
    ending_capital = weekly["equity_after"].iloc[-1]
    years = (pd.Timestamp(weekly["date"].iloc[-1]) - pd.Timestamp(weekly["date"].iloc[0])).days / 365.25
    cumulative_return = ending_capital / starting_capital - 1.0
    cagr_val = r10a.cagr(starting_capital, ending_capital, years)
    mdd = r10a.max_drawdown(returns)
    return {
        "starting_capital": starting_capital, "ending_capital": float(ending_capital),
        "cumulative_return": float(cumulative_return), "cagr": cagr_val,
        "annualized_volatility": r10a.annualized_volatility(returns),
        "sharpe_ratio": r10a.sharpe_ratio(returns), "sortino_ratio": r10a.sortino_ratio(returns),
        "max_drawdown": mdd, "calmar_ratio": r10a.calmar_ratio(cagr_val, mdd),
        "mean_weekly_return": float(np.mean(returns)), "median_weekly_return": float(np.median(returns)),
        "weekly_return_std": float(np.std(returns, ddof=0)),
        "best_week": float(np.max(returns)), "worst_week": float(np.min(returns)),
        "positive_week_rate": float((returns > 0).mean()), "n_weeks": len(weekly),
        "years_elapsed": years,
    }


def settle_from_ledger(ledger: pd.DataFrame, return_col: str, all_dates: list,
                        starting_capital: float = r10a.STARTING_CAPITAL) -> pd.DataFrame:
    """Every week in `all_dates` gets a row (0% return / cash if no trades that
    week -- halted or fully-ineligible weeks are NOT dropped)."""
    if len(ledger) > 0:
        by_week = ledger.groupby("date").apply(lambda g: (g["weight"] * g[return_col]).sum(),
                                                 include_groups=False)
    else:
        by_week = pd.Series(dtype=float)
    rows = [{"date": d, "portfolio_return": float(by_week.get(d, 0.0))} for d in all_dates]
    weekly = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    equity = r10a.equity_curve_from_returns(weekly["portfolio_return"].to_numpy(), starting_capital)
    weekly["equity_before"] = equity[:-1]; weekly["equity_after"] = equity[1:]
    return weekly


vti_dates_sorted = sorted(vti_weekly_for_metrics["date"].unique())
vti_core_metrics = compute_core_metrics(vti_weekly_for_metrics)

performance_rows = []
settlements_gross = {}
settlements_cost = {}
weekly_excess_store = {}
for key, ledger in all_ledgers.items():
    model_name, k, alloc, config_name = key
    all_dates = sorted(full_ranking[model_name]["date"].unique())
    weekly_gross = settle_from_ledger(ledger, "gross_trade_return", all_dates)
    weekly_cost = settle_from_ledger(ledger, "cost_adjusted_trade_return", all_dates)
    settlements_gross[key] = weekly_gross
    settlements_cost[key] = weekly_cost

    core_gross = compute_core_metrics(weekly_gross)
    core_cost = compute_core_metrics(weekly_cost)
    merged = weekly_gross.merge(vti_weekly_for_metrics[["date", "portfolio_return"]], on="date", suffixes=("", "_vti"))
    excess = merged["portfolio_return"] - merged["portfolio_return_vti"]
    weekly_excess_store[key] = merged.assign(excess_return=excess).set_index("date")["excess_return"]

    performance_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc, "filter_config": config_name,
        "gross_ending_capital": core_gross["ending_capital"], "gross_cagr": core_gross["cagr"],
        "gross_sharpe": core_gross["sharpe_ratio"], "gross_sortino": core_gross["sortino_ratio"],
        "gross_max_drawdown": core_gross["max_drawdown"], "gross_calmar": core_gross["calmar_ratio"],
        "gross_mean_weekly_return": core_gross["mean_weekly_return"], "gross_positive_week_rate": core_gross["positive_week_rate"],
        "cost_ending_capital": core_cost["ending_capital"], "cost_cagr": core_cost["cagr"],
        "cost_sharpe": core_cost["sharpe_ratio"], "cost_sortino": core_cost["sortino_ratio"],
        "cost_max_drawdown": core_cost["max_drawdown"],
        "cagr_drag_from_costs": core_gross["cagr"] - core_cost["cagr"],
        "mean_weekly_excess_return_vs_vti": float(excess.mean()),
        "excess_return_win_rate": float((excess > 0).mean()),
        "total_etf_trades": len(ledger), "n_trade_weeks": ledger["date"].nunique() if len(ledger) else 0,
        "mean_positions_per_week": float(ledger.groupby("date").size().mean()) if len(ledger) else 0.0,
        "n_weeks_total": len(all_dates),
    })
r10b_performance_summary = pd.DataFrame(performance_rows)
print(f"[{time.time()-t0:.0f}s] Step 8 (gross + cost-adjusted performance metrics, all 80 sims) done.")
print(r10b_performance_summary[r10b_performance_summary.filter_config.isin(["BASE", "ABCDE"])][
    ["model", "K", "allocation_method", "filter_config", "gross_cagr", "cost_cagr", "gross_max_drawdown"]
].to_string())

# ---------------------------------------------------------------------------

[70s] Step 8 (gross + cost-adjusted performance metrics, all 80 sims) done.
    model  K        allocation_method filter_config  gross_cagr  cost_cagr  gross_max_drawdown
0   NN_v1  2             EQUAL_WEIGHT          BASE    0.097116   0.041381           -0.614077
9   NN_v1  2             EQUAL_WEIGHT         ABCDE    0.060515   0.023846           -0.335021
10  NN_v1  2  PAPER_WEIGHT_NORMALIZED          BASE    0.095195   0.039557           -0.614530
19  NN_v1  2  PAPER_WEIGHT_NORMALIZED         ABCDE    0.065100   0.028221           -0.339387
20  NN_v1  3             EQUAL_WEIGHT          BASE    0.075929   0.021254           -0.614421
29  NN_v1  3             EQUAL_WEIGHT         ABCDE    0.054092   0.017916           -0.363465
30  NN_v1  3  PAPER_WEIGHT_NORMALIZED          BASE    0.076606   0.021898           -0.602500
39  NN_v1  3  PAPER_WEIGHT_NORMALIZED         ABCDE    0.051369   0.015284           -0.370189
40  NN_v2  2             EQUAL_WEIGHT          BASE    0.122992   0.0

## 9. Year-by-year performance, all 80 simulations.

In [10]:
# 9. Year-by-year performance (gross), all 80 sims.
# ---------------------------------------------------------------------------
yearly_rows = []
for key, weekly in settlements_gross.items():
    model_name, k, alloc, config_name = key
    w = weekly.copy(); w["year"] = pd.to_datetime(w["date"]).dt.year
    vti_y = vti_weekly_for_metrics.copy(); vti_y["year"] = pd.to_datetime(vti_y["date"]).dt.year
    ledger = all_ledgers[key].copy()
    if len(ledger):
        ledger["year"] = pd.to_datetime(ledger["date"]).dt.year
    for year, g in w.groupby("year"):
        vti_g = vti_y[vti_y["year"] == year]
        strat_year_return = float(np.prod(1 + g["portfolio_return"]) - 1)
        vti_year_return = float(np.prod(1 + vti_g["portfolio_return"]) - 1) if len(vti_g) else np.nan
        n_halted = sum(1 for d in g["date"] if len(ledger) == 0 or d not in set(ledger["date"]))
        yearly_rows.append({
            "model": model_name, "K": k, "allocation_method": alloc, "filter_config": config_name, "year": int(year),
            "strategy_return": strat_year_return, "vti_return": vti_year_return,
            "excess_return": strat_year_return - vti_year_return if not np.isnan(vti_year_return) else np.nan,
            "max_drawdown": r10a.max_drawdown(g["portfolio_return"].to_numpy()),
            "sharpe": r10a.sharpe_ratio(g["portfolio_return"].to_numpy()) if len(g) >= 2 else np.nan,
            "total_etf_trades": int(len(ledger[ledger["year"] == year])) if len(ledger) else 0,
            "halted_weeks": int(n_halted),
        })
r10b_yearly_performance = pd.DataFrame(yearly_rows)
print(f"[{time.time()-t0:.0f}s] Step 9 (year-by-year performance) done.")

# ---------------------------------------------------------------------------

[156s] Step 9 (year-by-year performance) done.


## 10. Rule trigger counts (Step 17), attributed per-rule even within cumulative combined runs.

In [11]:
# 10. Rule trigger counts (Step 17) -- from final engine state, cumulative
#     configs (each cumulative step's engine has exactly the rules active up
#     to that step, so e.g. the "ABCD" engine's D-trigger count is Rule D's
#     count when combined with A+B+C already active).
# ---------------------------------------------------------------------------
trigger_count_rows = []
for key, engine in all_engines.items():
    model_name, k, alloc, config_name = key
    active = FILTER_CONFIGS[config_name]
    a_events = engine.a_trigger_events; d_events = engine.d_trigger_events; e_events = engine.e_trigger_events
    b_events = engine.b_trigger_events; c_events = engine.c_trigger_events
    trigger_count_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc, "filter_config": config_name,
        "A_triggers": len(a_events) if "A" in active else 0,
        "A_unique_weeks": len({w for w, s in a_events}) if "A" in active else 0,
        "A_unique_symbols": len({s for w, s in a_events}) if "A" in active else 0,
        "B_triggers": len(b_events) if "B" in active else 0,
        "C_triggers": len(c_events) if "C" in active else 0,
        "total_halted_weeks": len(engine.halted_week_idx) if ({"B", "C"} & active) else 0,
        "D_triggers": len(d_events) if "D" in active else 0,
        "D_symbols_permanently_blocked": sum(engine.symbol_permanently_blocked.values()) if "D" in active else 0,
        "E_triggers": len(e_events) if "E" in active else 0,
        "E_unique_symbols": len({s for w, s in e_events}) if "E" in active else 0,
    })
r10b_rule_trigger_counts = pd.DataFrame(trigger_count_rows)
print(f"[{time.time()-t0:.0f}s] Step 10 (rule trigger counts) done.")
print(r10b_rule_trigger_counts[r10b_rule_trigger_counts.filter_config == "ABCDE"][
    ["model", "K", "allocation_method", "A_triggers", "total_halted_weeks", "D_symbols_permanently_blocked", "E_triggers"]
].to_string())

# ---------------------------------------------------------------------------

[156s] Step 10 (rule trigger counts) done.
    model  K        allocation_method  A_triggers  total_halted_weeks  D_symbols_permanently_blocked  E_triggers
9   NN_v1  2             EQUAL_WEIGHT          47                 315                              0         203
19  NN_v1  2  PAPER_WEIGHT_NORMALIZED          47                 314                              1         213
29  NN_v1  3             EQUAL_WEIGHT          60                 320                              2         125
39  NN_v1  3  PAPER_WEIGHT_NORMALIZED          60                 320                              2         125
49  NN_v2  2             EQUAL_WEIGHT          46                 311                              0         150
59  NN_v2  2  PAPER_WEIGHT_NORMALIZED          46                 310                              0         178
69  NN_v2  3             EQUAL_WEIGHT          58                 322                              0          37
79  NN_v2  3  PAPER_WEIGHT_NORMALIZED          58    

## 11. Turnover / transaction-cost effect (Step 18): does each filter naturally reduce trading frequency, and does that translate into reduced cost drag?

In [12]:
# 11. Turnover / transaction-cost effect (Step 18) -- per config, entries
#     per week, traded notional/equity, cost dollars, cost-drag reduction
#     relative to the SAME variant's own BASE.
# ---------------------------------------------------------------------------
turnover_rows = []
for key, ledger in all_ledgers.items():
    model_name, k, alloc, config_name = key
    n_weeks_total = len(sorted(full_ranking[model_name]["date"].unique()))
    n_trade_weeks = ledger["date"].nunique() if len(ledger) else 0
    mean_positions = float(ledger.groupby("date").size().mean()) if len(ledger) else 0.0
    # each traded position is fully bought AND sold every week it's held (weekly full rotation) ->
    # gross traded notional that week = 2 * equity (buy leg + sell leg), only on trade weeks.
    cost_dollars = float((ledger["weight"] * (ledger["gross_trade_return"] - ledger["cost_adjusted_trade_return"])).sum()) if len(ledger) else 0.0
    perf_row = r10b_performance_summary[(r10b_performance_summary.model == model_name) & (r10b_performance_summary.K == k) &
                                          (r10b_performance_summary.allocation_method == alloc) &
                                          (r10b_performance_summary.filter_config == config_name)].iloc[0]
    turnover_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc, "filter_config": config_name,
        "n_weeks_total": n_weeks_total, "n_trade_weeks": n_trade_weeks,
        "active_week_fraction": n_trade_weeks / n_weeks_total if n_weeks_total else np.nan,
        "mean_positions_per_trade_week": mean_positions,
        "gross_traded_notional_over_equity_per_trade_week": 2.0,
        "total_etf_entries": len(ledger), "total_etf_exits": len(ledger),
        "cost_adjusted_return_dollars_equiv_fraction": cost_dollars,
        "gross_to_cost_cagr_gap": perf_row["cagr_drag_from_costs"],
    })
r10b_turnover_cost_metrics = pd.DataFrame(turnover_rows)

turnover_reduction_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for k in K_VALUES:
        for alloc in r10a.ALLOCATION_METHODS:
            base = r10b_turnover_cost_metrics[(r10b_turnover_cost_metrics.model == model_name) & (r10b_turnover_cost_metrics.K == k) &
                                               (r10b_turnover_cost_metrics.allocation_method == alloc) &
                                               (r10b_turnover_cost_metrics.filter_config == "BASE")].iloc[0]
            for config_name in CUMULATIVE_CONFIG_NAMES:
                if config_name == "BASE":
                    continue
                row = r10b_turnover_cost_metrics[(r10b_turnover_cost_metrics.model == model_name) & (r10b_turnover_cost_metrics.K == k) &
                                                  (r10b_turnover_cost_metrics.allocation_method == alloc) &
                                                  (r10b_turnover_cost_metrics.filter_config == config_name)].iloc[0]
                turnover_reduction_rows.append({
                    "model": model_name, "K": k, "allocation_method": alloc, "filter_config": config_name,
                    "turnover_reduction": base["active_week_fraction"] - row["active_week_fraction"],
                    "cost_drag_reduction_cagr_pts": base["gross_to_cost_cagr_gap"] - row["gross_to_cost_cagr_gap"],
                })
r10b_turnover_reduction = pd.DataFrame(turnover_reduction_rows)
print(f"[{time.time()-t0:.0f}s] Step 11 (turnover / cost effect) done.")

# ---------------------------------------------------------------------------

[157s] Step 11 (turnover / cost effect) done.


## 12. Drawdown detail (Step 19): episode counts by severity, downside deviation, worst-week/4-week/year.

In [13]:
# 12. Drawdown detail (Step 19).
# ---------------------------------------------------------------------------
def drawdown_episode_metrics(returns: np.ndarray) -> dict:
    equity = r10a.equity_curve_from_returns(returns, starting_capital=1.0)
    running_max = np.maximum.accumulate(equity)
    dd = equity / running_max - 1.0
    in_dd = dd[1:] < 0  # align to return-index (dd has len+1 due to the seeded starting point)
    episodes = []
    cur_start = None
    for i, flag in enumerate(in_dd):
        if flag and cur_start is None:
            cur_start = i
        elif not flag and cur_start is not None:
            episodes.append((cur_start, i)); cur_start = None
    if cur_start is not None:
        episodes.append((cur_start, len(in_dd)))
    episode_max_dd = [dd[1:][a:b].min() for a, b in episodes] if episodes else []
    downside = returns[returns < 0]
    worst_4week = float(pd.Series(returns).rolling(4).apply(lambda x: np.prod(1 + x) - 1, raw=True).min()) if len(returns) >= 4 else np.nan
    return {
        "max_drawdown": r10a.max_drawdown(returns),
        "longest_drawdown_duration_weeks": max((b - a for a, b in episodes), default=0),
        "n_drawdowns_gt10pct": sum(1 for d in episode_max_dd if d <= -0.10),
        "n_drawdowns_gt20pct": sum(1 for d in episode_max_dd if d <= -0.20),
        "n_drawdowns_gt30pct": sum(1 for d in episode_max_dd if d <= -0.30),
        "downside_deviation": float(np.std(downside, ddof=0)) if len(downside) else np.nan,
        "worst_week": float(np.min(returns)), "worst_4week_rolling_return": worst_4week,
    }


drawdown_rows = []
for key, weekly in settlements_gross.items():
    model_name, k, alloc, config_name = key
    dd = drawdown_episode_metrics(weekly["portfolio_return"].to_numpy())
    w = weekly.copy(); w["year"] = pd.to_datetime(w["date"]).dt.year
    worst_year = w.groupby("year")["portfolio_return"].apply(lambda r: float(np.prod(1 + r) - 1)).min()
    drawdown_rows.append({"model": model_name, "K": k, "allocation_method": alloc, "filter_config": config_name,
                           **dd, "worst_calendar_year_return": float(worst_year)})
r10b_drawdown_metrics = pd.DataFrame(drawdown_rows)
print(f"[{time.time()-t0:.0f}s] Step 12 (drawdown detail) done.")

# ---------------------------------------------------------------------------

[157s] Step 12 (drawdown detail) done.


## 13. Stress-period diagnostic (Step 23): predeclared calendar years 2008 (financial crisis), 2020 (COVID shock), 2022 (bear market).

In [14]:
# 13. Stress-period diagnostic (Step 23) -- predeclared calendar years:
#     2008 (financial crisis), 2020 (COVID shock), 2022 (equity bear market).
# ---------------------------------------------------------------------------
STRESS_YEARS = {"2008_financial_crisis": 2008, "2020_covid_shock": 2020, "2022_bear_market": 2022}
stress_rows = []
for label, year in STRESS_YEARS.items():
    for model_name in ["NN_v1", "NN_v2"]:
        for k in K_VALUES:
            for alloc in r10a.ALLOCATION_METHODS:
                base = r10b_yearly_performance[(r10b_yearly_performance.model == model_name) & (r10b_yearly_performance.K == k) &
                                                (r10b_yearly_performance.allocation_method == alloc) &
                                                (r10b_yearly_performance.filter_config == "BASE") &
                                                (r10b_yearly_performance.year == year)]
                full = r10b_yearly_performance[(r10b_yearly_performance.model == model_name) & (r10b_yearly_performance.K == k) &
                                                (r10b_yearly_performance.allocation_method == alloc) &
                                                (r10b_yearly_performance.filter_config == "ABCDE") &
                                                (r10b_yearly_performance.year == year)]
                if len(base) == 0 or len(full) == 0:
                    continue
                stress_rows.append({
                    "stress_period": label, "year": year, "model": model_name, "K": k, "allocation_method": alloc,
                    "baseline_return": base["strategy_return"].iloc[0], "full_stack_return": full["strategy_return"].iloc[0],
                    "vti_return": base["vti_return"].iloc[0],
                    "baseline_max_drawdown": base["max_drawdown"].iloc[0], "full_stack_max_drawdown": full["max_drawdown"].iloc[0],
                    "full_stack_halted_weeks": full["halted_weeks"].iloc[0],
                })
r10b_stress_period_metrics = pd.DataFrame(stress_rows)
print(f"[{time.time()-t0:.0f}s] Step 13 (stress-period diagnostic) done.")

# ---------------------------------------------------------------------------

[157s] Step 13 (stress-period diagnostic) done.


## 14. VTI comparison (gross), all 80 simulations.

In [15]:
# 14. VTI comparison (gross), all 80 sims.
# ---------------------------------------------------------------------------
vs_vti_rows = []
for key, excess in weekly_excess_store.items():
    model_name, k, alloc, config_name = key
    perf = r10b_performance_summary[(r10b_performance_summary.model == model_name) & (r10b_performance_summary.K == k) &
                                     (r10b_performance_summary.allocation_method == alloc) &
                                     (r10b_performance_summary.filter_config == config_name)].iloc[0]
    vs_vti_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc, "filter_config": config_name,
        "cumulative_return_minus_vti": (perf["gross_ending_capital"] / r10a.STARTING_CAPITAL - 1) - vti_core_metrics["cumulative_return"],
        "cagr_minus_vti": perf["gross_cagr"] - vti_core_metrics["cagr"],
        "mean_weekly_excess_return": float(excess.mean()), "excess_return_win_rate": float((excess > 0).mean()),
        "tracking_error": r10a.tracking_error(excess.to_numpy()), "information_ratio": r10a.information_ratio(excess.to_numpy()),
    })
r10b_vs_vti_metrics = pd.DataFrame(vs_vti_rows)
print(f"[{time.time()-t0:.0f}s] Step 14 (VTI comparison) done.")

# ---------------------------------------------------------------------------

[157s] Step 14 (VTI comparison) done.


## 15. Equal vs. paper weight, after risk filters (Step 24), across all 10 filter configurations.

In [16]:
# 15. Equal vs paper weight, after risk filters (Step 24), all filter configs.
# ---------------------------------------------------------------------------
block_bootstrap_ci = r7.block_bootstrap_ci
equal_vs_paper_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for k in K_VALUES:
        for config_name in FILTER_CONFIGS:
            eq_key = (model_name, k, "EQUAL_WEIGHT", config_name)
            pw_key = (model_name, k, "PAPER_WEIGHT_NORMALIZED", config_name)
            eq = settlements_gross[eq_key].set_index("date"); pw = settlements_gross[pw_key].set_index("date")
            common = eq.index.intersection(pw.index)
            diff = (pw.loc[common, "portfolio_return"] - eq.loc[common, "portfolio_return"]).sort_index()
            b = block_bootstrap_ci(diff) if len(diff) >= r7.BLOCK_SIZE else None
            eq_perf = r10b_performance_summary[(r10b_performance_summary.model == model_name) & (r10b_performance_summary.K == k) &
                                                (r10b_performance_summary.allocation_method == "EQUAL_WEIGHT") &
                                                (r10b_performance_summary.filter_config == config_name)].iloc[0]
            pw_perf = r10b_performance_summary[(r10b_performance_summary.model == model_name) & (r10b_performance_summary.K == k) &
                                                (r10b_performance_summary.allocation_method == "PAPER_WEIGHT_NORMALIZED") &
                                                (r10b_performance_summary.filter_config == config_name)].iloc[0]
            row = {
                "model": model_name, "K": k, "filter_config": config_name,
                "cagr_diff_paper_minus_equal": pw_perf["gross_cagr"] - eq_perf["gross_cagr"],
                "sharpe_diff": pw_perf["gross_sharpe"] - eq_perf["gross_sharpe"],
                "max_drawdown_diff": pw_perf["gross_max_drawdown"] - eq_perf["gross_max_drawdown"],
            }
            if b is not None:
                row.update({"mean_weekly_return_diff_point": b["point_estimate"], "ci_lower": b["ci_lower"],
                            "ci_upper": b["ci_upper"], "significant": b["significant"]})
            equal_vs_paper_rows.append(row)
r10b_equal_vs_paper_comparison = pd.DataFrame(equal_vs_paper_rows)
print(f"[{time.time()-t0:.0f}s] Step 15 (equal vs paper, all filter configs) done.")

# ---------------------------------------------------------------------------

[164s] Step 15 (equal vs paper, all filter configs) done.


## 16. Top-2 vs. Top-3 descriptive comparison (Step 25) -- no statistical test, no K optimization.

In [17]:
# 16. Top-2 vs Top-3 descriptive comparison (Step 25) -- no statistical test.
# ---------------------------------------------------------------------------
top2_vs_top3_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for alloc in r10a.ALLOCATION_METHODS:
        for config_name in FILTER_CONFIGS:
            r2 = r10b_performance_summary[(r10b_performance_summary.model == model_name) & (r10b_performance_summary.K == 2) &
                                           (r10b_performance_summary.allocation_method == alloc) &
                                           (r10b_performance_summary.filter_config == config_name)].iloc[0]
            r3 = r10b_performance_summary[(r10b_performance_summary.model == model_name) & (r10b_performance_summary.K == 3) &
                                           (r10b_performance_summary.allocation_method == alloc) &
                                           (r10b_performance_summary.filter_config == config_name)].iloc[0]
            top2_vs_top3_rows.append({
                "model": model_name, "allocation_method": alloc, "filter_config": config_name,
                "top2_gross_cagr": r2["gross_cagr"], "top3_gross_cagr": r3["gross_cagr"],
                "top2_max_drawdown": r2["gross_max_drawdown"], "top3_max_drawdown": r3["gross_max_drawdown"],
                "top2_sharpe": r2["gross_sharpe"], "top3_sharpe": r3["gross_sharpe"],
            })
r10b_top2_vs_top3_descriptive = pd.DataFrame(top2_vs_top3_rows)
print(f"[{time.time()-t0:.0f}s] Step 16 (Top-2 vs Top-3 descriptive) done.")

# ---------------------------------------------------------------------------

[164s] Step 16 (Top-2 vs Top-3 descriptive) done.


## 17. Block bootstrap: full risk stack (ABCDE) vs. the same variant's own no-filter BASE.

In [18]:
# 17. Block bootstrap: full stack (ABCDE) vs same-variant BASE.
# ---------------------------------------------------------------------------
bootstrap_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for k in K_VALUES:
        for alloc in r10a.ALLOCATION_METHODS:
            base_key = (model_name, k, alloc, "BASE")
            full_key = (model_name, k, alloc, "ABCDE")
            base_w = settlements_gross[base_key].set_index("date"); full_w = settlements_gross[full_key].set_index("date")
            common = base_w.index.intersection(full_w.index)
            diff_ret = (full_w.loc[common, "portfolio_return"] - base_w.loc[common, "portfolio_return"]).sort_index()
            b = block_bootstrap_ci(diff_ret)
            bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc,
                                    "quantity": "mean_weekly_return_diff_fullstack_minus_base", **b})

            base_excess = weekly_excess_store[base_key]; full_excess = weekly_excess_store[full_key]
            common_e = base_excess.index.intersection(full_excess.index)
            diff_excess = (full_excess.loc[common_e] - base_excess.loc[common_e]).sort_index()
            b = block_bootstrap_ci(diff_excess)
            bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc,
                                    "quantity": "mean_weekly_excess_vti_diff_fullstack_minus_base", **b})

            full_w_full = settlements_gross[full_key]
            b = block_bootstrap_ci(full_w_full.set_index("date")["portfolio_return"])
            bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc,
                                    "quantity": "fullstack_mean_weekly_return_vs_0", **b})
            b = block_bootstrap_ci(weekly_excess_store[full_key])
            bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc,
                                    "quantity": "fullstack_mean_weekly_excess_vti_vs_0", **b})

            pos_week_base = (base_w.loc[common, "portfolio_return"] > 0).astype(float)
            pos_week_full = (full_w.loc[common, "portfolio_return"] > 0).astype(float)
            b = block_bootstrap_ci((pos_week_full - pos_week_base).sort_index())
            bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc,
                                    "quantity": "positive_week_rate_diff_fullstack_minus_base", **b})
r10b_bootstrap_ci = pd.DataFrame(bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 17 (block bootstrap, full stack vs base) done.")
print(r10b_bootstrap_ci[r10b_bootstrap_ci.quantity == "mean_weekly_return_diff_fullstack_minus_base"][
    ["model", "K", "allocation_method", "point_estimate", "significant"]].to_string())

# ---------------------------------------------------------------------------

[171s] Step 17 (block bootstrap, full stack vs base) done.
    model  K        allocation_method  point_estimate  significant
0   NN_v1  2             EQUAL_WEIGHT       -0.000881        False
5   NN_v1  2  PAPER_WEIGHT_NORMALIZED       -0.000756        False
10  NN_v1  3             EQUAL_WEIGHT       -0.000609        False
15  NN_v1  3  PAPER_WEIGHT_NORMALIZED       -0.000669        False
20  NN_v2  2             EQUAL_WEIGHT       -0.000428        False
25  NN_v2  2  PAPER_WEIGHT_NORMALIZED       -0.000459        False
30  NN_v2  3             EQUAL_WEIGHT       -0.000232        False
35  NN_v2  3  PAPER_WEIGHT_NORMALIZED       -0.000224        False


## 18. Integrated alpha status (Step 29) for the full risk stack, per (model, K, allocation).

In [19]:
# 18. Full-stack (ABCDE) integrated alpha status per (model, K, allocation).
# ---------------------------------------------------------------------------
def classify_alpha_status(gross_cagr_gap, cost_cagr, vti_cagr, mean_excess, excess_significant,
                           sharpe, max_dd, vti_max_dd) -> str:
    positives = 0; negatives = 0
    positives += 1 if gross_cagr_gap > 0 else -1
    positives += 1 if cost_cagr > vti_cagr else -1
    positives += 1 if mean_excess > 0 else -1
    positives += 1 if max_dd > vti_max_dd else -1  # less negative = better
    if excess_significant and mean_excess > 0:
        return "ALPHA_POSITIVE"
    if excess_significant and mean_excess < 0:
        return "ALPHA_NEGATIVE"
    if positives >= 2:
        return "ALPHA_MIXED" if positives < 4 else "ALPHA_POSITIVE"
    if positives <= -2:
        return "ALPHA_MIXED" if positives > -4 else "ALPHA_NEGATIVE"
    return "ALPHA_NULL"


alpha_status_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for k in K_VALUES:
        for alloc in r10a.ALLOCATION_METHODS:
            perf = r10b_performance_summary[(r10b_performance_summary.model == model_name) & (r10b_performance_summary.K == k) &
                                             (r10b_performance_summary.allocation_method == alloc) &
                                             (r10b_performance_summary.filter_config == "ABCDE")].iloc[0]
            vs_vti = r10b_vs_vti_metrics[(r10b_vs_vti_metrics.model == model_name) & (r10b_vs_vti_metrics.K == k) &
                                          (r10b_vs_vti_metrics.allocation_method == alloc) &
                                          (r10b_vs_vti_metrics.filter_config == "ABCDE")].iloc[0]
            dd = r10b_drawdown_metrics[(r10b_drawdown_metrics.model == model_name) & (r10b_drawdown_metrics.K == k) &
                                        (r10b_drawdown_metrics.allocation_method == alloc) &
                                        (r10b_drawdown_metrics.filter_config == "ABCDE")].iloc[0]
            boot = r10b_bootstrap_ci[(r10b_bootstrap_ci.model == model_name) & (r10b_bootstrap_ci.K == k) &
                                      (r10b_bootstrap_ci.allocation_method == alloc) &
                                      (r10b_bootstrap_ci.quantity == "fullstack_mean_weekly_excess_vti_vs_0")].iloc[0]
            status = classify_alpha_status(
                gross_cagr_gap=vs_vti["cagr_minus_vti"], cost_cagr=perf["cost_cagr"], vti_cagr=vti_core_metrics["cagr"],
                mean_excess=vs_vti["mean_weekly_excess_return"], excess_significant=bool(boot["significant"]),
                sharpe=perf["gross_sharpe"], max_dd=dd["max_drawdown"], vti_max_dd=vti_core_metrics["max_drawdown"],
            )
            alpha_status_rows.append({
                "model": model_name, "K": k, "allocation_method": alloc, "filter_config": "ABCDE",
                "gross_cagr": perf["gross_cagr"], "cost_cagr": perf["cost_cagr"], "vti_cagr": vti_core_metrics["cagr"],
                "mean_weekly_excess_return": vs_vti["mean_weekly_excess_return"],
                "excess_vs_0_significant": bool(boot["significant"]), "sharpe": perf["gross_sharpe"],
                "max_drawdown": dd["max_drawdown"], "vti_max_drawdown": vti_core_metrics["max_drawdown"],
                "cagr_drag_from_costs": perf["cagr_drag_from_costs"], "alpha_status": status,
            })
r10b_alpha_status = pd.DataFrame(alpha_status_rows)
print(f"[{time.time()-t0:.0f}s] Step 18 (integrated alpha status) done.")
print(r10b_alpha_status[["model", "K", "allocation_method", "alpha_status"]].to_string())

# ---------------------------------------------------------------------------

[171s] Step 18 (integrated alpha status) done.
   model  K        allocation_method alpha_status
0  NN_v1  2             EQUAL_WEIGHT  ALPHA_MIXED
1  NN_v1  2  PAPER_WEIGHT_NORMALIZED  ALPHA_MIXED
2  NN_v1  3             EQUAL_WEIGHT  ALPHA_MIXED
3  NN_v1  3  PAPER_WEIGHT_NORMALIZED  ALPHA_MIXED
4  NN_v2  2             EQUAL_WEIGHT  ALPHA_MIXED
5  NN_v2  2  PAPER_WEIGHT_NORMALIZED  ALPHA_MIXED
6  NN_v2  3             EQUAL_WEIGHT   ALPHA_NULL
7  NN_v2  3  PAPER_WEIGHT_NORMALIZED   ALPHA_NULL


## 19. Rule source audit table (Step 1) -- Table 2 of Bock & Maewal (SSRN 4280640), verbatim wording + confidence classification + documented unresolved ambiguity.

In [20]:
# 19. Rule source audit table (Step 1) -- Table 2 of Bock & Maewal (SSRN
#     4280640), verbatim wording + confidence classification + documented
#     unresolved ambiguity for each rule's reconstruction.
# ---------------------------------------------------------------------------
rule_audit_rows = [
    {
        "rule": "A", "label": r10b.FILTER_LABELS["A"], "paper_condition": "Recent loss by symbol",
        "paper_value": "5%", "paper_time": "week", "paper_action": "Remove ETF from buy list",
        "affected_scope": "individual symbol", "reconstructed_duration": "1 week (immediately following week only)",
        "restart_condition": "automatic, after exactly 1 week",
        "confidence": "PAPER_EXPLICIT_THRESHOLD_BUT_STATE_AMBIGUOUS",
        "unresolved_ambiguity": "Exact block duration is not stated in the paper; 1 week is the minimal, "
                                 "bounded reconstruction consistent with the paper's own weekly decision cadence.",
    },
    {
        "rule": "B", "label": r10b.FILTER_LABELS["B"], "paper_condition": "Maximum loss, week-week",
        "paper_value": "$300", "paper_time": "week", "paper_action": "Halt trading one week",
        "affected_scope": "whole portfolio", "reconstructed_duration": "1 week (explicit in source)",
        "restart_condition": "automatic, after exactly 1 week",
        "confidence": "PAPER_EXPLICIT_THRESHOLD_BUT_STATE_AMBIGUOUS",
        "unresolved_ambiguity": "Threshold ($300) and halt duration (1 week) are explicit; the precise quantity "
                                 "measured by \"week-week\" loss is not spelled out -- reconstructed as realized "
                                 "weekly portfolio dollar P&L (equity_after - equity_before for that trading week), "
                                 "the most literal reading of a week-over-week comparison.",
    },
    {
        "rule": "C", "label": r10b.FILTER_LABELS["C"], "paper_condition": "Portfolio underwater",
        "paper_value": "5%", "paper_time": "Q4", "paper_action": "Halt trading one week",
        "affected_scope": "whole portfolio", "reconstructed_duration": "1 week (explicit in source)",
        "restart_condition": "automatic, after exactly 1 week",
        "confidence": "PAPER_EXPLICIT_THRESHOLD_BUT_STATE_AMBIGUOUS",
        "unresolved_ambiguity": "Threshold (5%), scope (Q4 only), and halt duration (1 week) are explicit; the "
                                 "reference level for \"underwater\" is not stated. Reconstructed as equity at or "
                                 "below 95% of the equity level at the START of that year's Q4 (beginning-of-quarter "
                                 "reference, not a high-water mark, per the mandate's preferred reconstruction).",
    },
    {
        "rule": "D", "label": r10b.FILTER_LABELS["D"], "paper_condition": "Maximum loss by symbol",
        "paper_value": "27.5%", "paper_time": "Q1-Q4", "paper_action": "Remove ETF from buy list",
        "affected_scope": "individual symbol", "reconstructed_duration": "permanent (no re-entry condition stated)",
        "restart_condition": "none -- the paper gives no re-entry condition, and none is invented",
        "confidence": "PAPER_EXPLICIT_THRESHOLD_BUT_STATE_AMBIGUOUS",
        "unresolved_ambiguity": "Threshold (27.5%) and scope (all year, \"Q1-Q4\") are explicit; whether this is a "
                                 "single-trade or cumulative measure is not stated -- a 27.5% single-week ETF loss "
                                 "would be extraordinarily rare, so reconstructed as cumulative compounded return "
                                 "since the symbol's first completed trade in the evaluation period. Duration is "
                                 "reconstructed as permanent, since the paper specifies no reset/re-entry rule.",
    },
    {
        "rule": "E", "label": r10b.FILTER_LABELS["E"], "paper_condition": "Minimum win rate by symbol",
        "paper_value": "45%", "paper_time": "Q4", "paper_action": "Remove ETF from buy list",
        "affected_scope": "individual symbol", "reconstructed_duration": "active only during Q4 weeks, re-evaluated weekly",
        "restart_condition": "automatic outside Q4 (rule simply does not apply Jan-Sep)",
        "confidence": "PAPER_EXPLICIT_THRESHOLD_BUT_STATE_AMBIGUOUS",
        "unresolved_ambiguity": "Threshold (45%) and scope (Q4 only) are explicit; whether \"win rate\" means "
                                 "lifetime-cumulative or Q4-only trades, and any minimum sample size, are not "
                                 "stated. Reconstructed as lifetime cumulative win rate (identical wins/buys "
                                 "definition already used by the paper's own allocation formula), checked only "
                                 "during Q4; a symbol with zero prior trades is never excluded (no basis to judge).",
    },
]
r10b_rule_source_audit = pd.DataFrame(rule_audit_rows)
print(f"[{time.time()-t0:.0f}s] Step 19 (rule source audit table) done.")

# ---------------------------------------------------------------------------

[171s] Step 19 (rule source audit table) done.


## 20. Sample audit.

In [21]:
# 20. Sample audit.
# ---------------------------------------------------------------------------
sample_audit_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    oof = OOF_BY_MODEL[model_name]
    sample_audit_rows.append({
        "model": model_name, "n_oos_rows": len(oof), "n_weeks": oof["date"].nunique(),
        "first_date": str(oof["date"].min().date()), "last_date": str(oof["date"].max().date()),
        "underlying_auc": float(roc_auc_score(oof.label_binary, oof.predicted_proba)),
        "starting_capital": r10a.STARTING_CAPITAL, "cost_bps_round_trip": r10a.COST_BPS_PER_BUY + r10a.COST_BPS_PER_SELL,
        "n_filter_configs_tested": len(FILTER_CONFIGS), "n_total_simulations": len(all_ledgers),
    })
r10b_sample_audit = pd.DataFrame(sample_audit_rows)

# ---------------------------------------------------------------------------

## 21. Save all required R10B artifacts.

In [22]:
# 21. Save all artifacts.
# ---------------------------------------------------------------------------
r10b_rule_source_audit.to_csv(OUTPUTS_ROOT / "r10b_rule_source_audit.csv", index=False)

individual_metrics = r10b_performance_summary[r10b_performance_summary.filter_config.isin(INDIVIDUAL_CONFIG_NAMES)]
individual_metrics.to_csv(OUTPUTS_ROOT / "r10b_individual_filter_metrics.csv", index=False)
cumulative_metrics = r10b_performance_summary[r10b_performance_summary.filter_config.isin(CUMULATIVE_CONFIG_NAMES)]
cumulative_metrics.to_csv(OUTPUTS_ROOT / "r10b_cumulative_filter_metrics.csv", index=False)

trigger_log_frames = []
for key, tlog in all_trigger_logs.items():
    model_name, k, alloc, config_name = key
    t = tlog.copy(); t.insert(0, "model", model_name); t.insert(1, "K", k); t.insert(2, "allocation_method", alloc)
    t.insert(3, "filter_config", config_name)
    trigger_log_frames.append(t)
r10b_filter_trigger_log = pd.concat(trigger_log_frames, ignore_index=True)
r10b_filter_trigger_log.to_csv(OUTPUTS_ROOT / "r10b_filter_trigger_log.csv", index=False)

weekly_positions_frames = []
trade_ledger_frames = []
for key, ledger in all_ledgers.items():
    model_name, k, alloc, config_name = key
    lg = ledger.copy(); lg.insert(0, "model", model_name); lg.insert(1, "K", k); lg.insert(2, "allocation_method", alloc)
    lg.insert(3, "filter_config", config_name)
    trade_ledger_frames.append(lg)
    if len(ledger):
        pos = ledger[["date", "symbol", "rank", "weight"]].copy()
        pos.insert(0, "model", model_name); pos.insert(1, "K", k); pos.insert(2, "allocation_method", alloc)
        pos.insert(3, "filter_config", config_name)
        weekly_positions_frames.append(pos)
r10b_trade_ledger = pd.concat(trade_ledger_frames, ignore_index=True)
r10b_trade_ledger.to_csv(OUTPUTS_ROOT / "r10b_trade_ledger.csv", index=False)
r10b_weekly_positions = pd.concat(weekly_positions_frames, ignore_index=True)
r10b_weekly_positions.to_csv(OUTPUTS_ROOT / "r10b_weekly_positions.csv", index=False)

weekly_returns_frames = []
equity_curve_frames = []
for key, weekly in settlements_gross.items():
    model_name, k, alloc, config_name = key
    w = weekly.copy(); w.insert(0, "model", model_name); w.insert(1, "K", k); w.insert(2, "allocation_method", alloc)
    w.insert(3, "filter_config", config_name)
    weekly_returns_frames.append(w)
r10b_weekly_portfolio_returns = pd.concat(weekly_returns_frames, ignore_index=True)
r10b_weekly_portfolio_returns.to_csv(OUTPUTS_ROOT / "r10b_weekly_portfolio_returns.csv", index=False)

vti_curve_row = vti_weekly_for_metrics.copy()
vti_curve_row.insert(0, "model", "VTI_BENCHMARK"); vti_curve_row.insert(1, "K", np.nan)
vti_curve_row.insert(2, "allocation_method", "N/A"); vti_curve_row.insert(3, "filter_config", "N/A")
r10b_equity_curves = pd.concat([r10b_weekly_portfolio_returns, vti_curve_row], ignore_index=True)
r10b_equity_curves.to_csv(OUTPUTS_ROOT / "r10b_equity_curves.csv", index=False)

r10b_performance_summary.to_csv(OUTPUTS_ROOT / "r10b_performance_summary.csv", index=False)
r10b_yearly_performance.to_csv(OUTPUTS_ROOT / "r10b_yearly_performance.csv", index=False)
r10b_drawdown_metrics.to_csv(OUTPUTS_ROOT / "r10b_drawdown_metrics.csv", index=False)
r10b_turnover_cost_metrics.to_csv(OUTPUTS_ROOT / "r10b_turnover_cost_metrics.csv", index=False)
r10b_turnover_reduction.to_csv(OUTPUTS_ROOT / "r10b_turnover_reduction.csv", index=False)
r10b_vs_vti_metrics.to_csv(OUTPUTS_ROOT / "r10b_vs_vti_metrics.csv", index=False)
r10b_equal_vs_paper_comparison.to_csv(OUTPUTS_ROOT / "r10b_equal_vs_paper_comparison.csv", index=False)
r10b_top2_vs_top3_descriptive.to_csv(OUTPUTS_ROOT / "r10b_top2_vs_top3_descriptive.csv", index=False)
r10b_rule_trigger_counts.to_csv(OUTPUTS_ROOT / "r10b_rule_trigger_counts.csv", index=False)
r10b_stress_period_metrics.to_csv(OUTPUTS_ROOT / "r10b_stress_period_metrics.csv", index=False)
r10b_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r10b_bootstrap_ci.csv", index=False)
r10b_alpha_status.to_csv(OUTPUTS_ROOT / "r10b_alpha_status.csv", index=False)
r10b_sample_audit.to_csv(OUTPUTS_ROOT / "r10b_sample_audit.csv", index=False)

summary = {
    "starting_capital": r10a.STARTING_CAPITAL, "vti_cagr": vti_core_metrics["cagr"],
    "vti_max_drawdown": vti_core_metrics["max_drawdown"],
    "underlying_auc": {"NN_v1": float(auc_v1), "NN_v2": float(auc_v2)},
    "base_reproduces_r10a": True,
    "filter_configs_tested": list(FILTER_CONFIGS.keys()),
    "gross_cagr_base_vs_fullstack": {
        f"{m}_{k}_{a}": {"base": float(r10b_performance_summary[(r10b_performance_summary.model == m) & (r10b_performance_summary.K == k) &
                                                                   (r10b_performance_summary.allocation_method == a) &
                                                                   (r10b_performance_summary.filter_config == "BASE")]["gross_cagr"].iloc[0]),
                          "fullstack": float(r10b_performance_summary[(r10b_performance_summary.model == m) & (r10b_performance_summary.K == k) &
                                                                        (r10b_performance_summary.allocation_method == a) &
                                                                        (r10b_performance_summary.filter_config == "ABCDE")]["gross_cagr"].iloc[0])}
        for m in ["NN_v1", "NN_v2"] for k in K_VALUES for a in r10a.ALLOCATION_METHODS
    },
    "max_drawdown_base_vs_fullstack": {
        f"{m}_{k}_{a}": {"base": float(r10b_drawdown_metrics[(r10b_drawdown_metrics.model == m) & (r10b_drawdown_metrics.K == k) &
                                                                (r10b_drawdown_metrics.allocation_method == a) &
                                                                (r10b_drawdown_metrics.filter_config == "BASE")]["max_drawdown"].iloc[0]),
                          "fullstack": float(r10b_drawdown_metrics[(r10b_drawdown_metrics.model == m) & (r10b_drawdown_metrics.K == k) &
                                                                     (r10b_drawdown_metrics.allocation_method == a) &
                                                                     (r10b_drawdown_metrics.filter_config == "ABCDE")]["max_drawdown"].iloc[0])}
        for m in ["NN_v1", "NN_v2"] for k in K_VALUES for a in r10a.ALLOCATION_METHODS
    },
    "alpha_status_fullstack": {f"{r['model']}_{r['K']}_{r['allocation_method']}": r["alpha_status"] for r in alpha_status_rows},
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r10b_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2, default=str))

[173s] ALL DONE.
{
  "starting_capital": 100000.0,
  "vti_cagr": 0.07745027188070952,
  "vti_max_drawdown": -0.5532258982424447,
  "underlying_auc": {
    "NN_v1": 0.5693678469913763,
    "NN_v2": 0.5708322204557499
  },
  "base_reproduces_r10a": true,
  "filter_configs_tested": [
    "BASE",
    "A",
    "B",
    "C",
    "D",
    "E",
    "AB",
    "ABC",
    "ABCD",
    "ABCDE"
  ],
  "gross_cagr_base_vs_fullstack": {
    "NN_v1_2_EQUAL_WEIGHT": {
      "base": 0.0971161951830084,
      "fullstack": 0.060514522589004915
    },
    "NN_v1_2_PAPER_WEIGHT_NORMALIZED": {
      "base": 0.09519533714498118,
      "fullstack": 0.0650997159062896
    },
    "NN_v1_3_EQUAL_WEIGHT": {
      "base": 0.07592875583526615,
      "fullstack": 0.05409184339032036
    },
    "NN_v1_3_PAPER_WEIGHT_NORMALIZED": {
      "base": 0.0766057915081293,
      "fullstack": 0.05136857111422888
    },
    "NN_v2_2_EQUAL_WEIGHT": {
      "base": 0.12299158805520771,
      "fullstack": 0.10965790916350349
    },
